In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:

# ============================================
# Training & Inference EfficientNet untuk Rumah Adat Classification
# ============================================

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from sklearn.metrics import f1_score
import numpy as np
import pandas as pd
from tqdm import tqdm

# -----------------------------
# 1. Setup transformasi data
# -----------------------------
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # ukuran input EfficientNet
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# -----------------------------
# 2. Load dataset Train & split validasi
# -----------------------------
train_dir = "/kaggle/input/dsc-logika-ui-2025/Train/Train"
dataset = datasets.ImageFolder(root=train_dir, transform=transform)
class_names = dataset.classes
print("Kelas:", class_names)

val_size = int(0.2 * len(dataset))
train_size = len(dataset) - val_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# -----------------------------
# 3. Model EfficientNet-B3
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.efficientnet_b3(weights="IMAGENET1K_V1")
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, len(class_names))
model = model.to(device)

# Loss dan optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0003)

# -----------------------------
# 4. Training loop dengan F1 Score
# -----------------------------
num_epochs = 5
best_val_f1 = 0.0
best_model_path = "best_model.pth"

for epoch in range(num_epochs):
    # ---- Training ----
    model.train()
    running_loss, running_corrects = 0.0, 0
    all_preds, all_labels = [], []
    
    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"):
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        _, preds = torch.max(outputs, 1)
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    epoch_loss = running_loss / train_size
    epoch_acc = running_corrects.double() / train_size
    epoch_f1 = f1_score(all_labels, all_preds, average="macro")
    
    # ---- Validation ----
    model.eval()
    val_loss, val_corrects = 0.0, 0
    val_preds, val_labels = [], []
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            _, preds = torch.max(outputs, 1)
            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
    
    val_epoch_loss = val_loss / val_size
    val_epoch_acc = val_corrects.double() / val_size
    val_epoch_f1 = f1_score(val_labels, val_preds, average="macro")
    
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} F1: {epoch_f1:.4f}")
    print(f"  Val   Loss: {val_epoch_loss:.4f} Acc: {val_epoch_acc:.4f} F1: {val_epoch_f1:.4f}")
    
    # Simpan model terbaik berdasarkan F1 val
    if val_epoch_f1 > best_val_f1:
        best_val_f1 = val_epoch_f1
        torch.save(model.state_dict(), best_model_path)
        print("  ✅ Model disimpan (F1 val terbaik)")

# -----------------------------
# 5. Inference pada dataset Test
# -----------------------------
# Load kembali model terbaik
model.load_state_dict(torch.load(best_model_path))
model.eval()

test_dir = "/kaggle/input/dsc-logika-ui-2025/Test/Test"
test_images = sorted(os.listdir(test_dir))  # urutkan biar sesuai

predictions = []
ids = []

with torch.no_grad():
    for img_name in tqdm(test_images, desc="Predicting Test"):
        img_path = os.path.join(test_dir, img_name)
        
        # load image
        img = datasets.folder.default_loader(img_path)  # pakai PIL loader
        img = transform(img).unsqueeze(0).to(device)
        
        # prediksi
        outputs = model(img)
        _, pred = torch.max(outputs, 1)
        pred_class = class_names[pred.item()]
        
        # id = nama file tanpa .jpg
        img_id = os.path.splitext(img_name)[0]
        ids.append(img_id)
        predictions.append(pred_class)

# -----------------------------
# 6. Simpan ke CSV
# -----------------------------
df = pd.DataFrame({"id": ids, "style": predictions})
df = df.sort_values(by="id")  # pastikan urut
df.to_csv("submission.csv", index=False)

print("✅ Hasil prediksi disimpan ke submission.csv")
